# Chapter 0 — The Raw LLM Call

**Promise:** before loops, tools, or memory, there is one call.
You send a list of messages; the model sends one response back.
Everything in this curriculum is machinery around this call.

> **PRINCIPLE**
> The model is stateless. The harness owns the conversation.

**Thread so far:** nothing yet — our agent, **Tab**, who runs
The Morning Bell café, starts as one raw call. This chapter
gives Tab the call itself.

**One day, on purpose:** the whole curriculum replays a
single Day at the café, from 06:30 Open + ovens on to 17:30
Close + count the till. You will live this day sixteen times. Tab gets better. So will you.

**Time:** about 28 minutes · 4 stages

**Ladder:** ▶ call · loop · tools · context · memory · permissions · subagents · tracing · MCP · skills · safety · evals · hooks · planning · scheduling

Everything here runs on the deterministic fake model: $0, no API
key, no flakes. The same script gives the same response every run.

## The setup — The Morning Bell

You own a tiny café, **The Morning Bell**. Your counter help is
**Tab** — the agent you build in this course, one layer per
chapter, starting from the single call in this chapter.

The regulars, whose cards live by the register:

- **Sasha** — oat-milk latte, window seat.
- **Mina** — nine years old, here with the school group. Severe
  peanut allergy. The red card by the counter — *no peanuts at
  the prep counter, ever* — exists because of her.
- **Aldo** — decaf, the crossword, pays cash.
- **Priya** — the office order, Fridays.

Every chapter replays the same Day — 06:30 open to 17:30 close,
same day sheet, same regulars, same rushes. The Day never
changes, so when Tab's behaviour changes, your harness is what
changed. And a café is honest ground for learning this: stock,
money, and the queue can all be counted exactly, and every
mistake has a cost you can see. That is how this course can
grade you exactly instead of by feel.

## Contents

Four stages, in order — about 28 minutes in total. Each stage opens with its goal and closes with what you have.

1. [The failure, shown](#the-failure-shown) — the whole game first, then the starter's two bugs · Stage 1 · ~13 min
2. [Build the layer](#the-mechanism) — the messages shape, and an honest return · Stage 2 · ~4 min
3. [Sabotage and checkpoint](#sabotage) — forget on purpose; then the real tests · Stage 3 · ~6 min
4. [In the wild](#in-the-wild) — the same call inside real products · Stage 4 · ~5 min

**How this notebook works:** run cells top to bottom · answer PREDICT before running the reveal · hints and the solution stay folded until you have tried.

In [1]:
# Setup: find the repo root, then import the shared pieces.
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "shared" / "fake_model.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import importlib.util


def load_harness(variant):
    spec = importlib.util.spec_from_file_location(
        f"harness_{variant}", ROOT / f"chapters/00-raw-llm-call/{variant}/harness.py")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


sol = load_harness("solution")
starter = load_harness("starter")
from shared.fake_model import FakeModel, call, say  # the scripted model

## The whole game

**Goal:** watch the finished agent run once, end to end — every later chapter takes this one run apart.

Below is Tab in miniature — the agent this curriculum ends with — doing its thread task on the fake model:
- read `day.txt`, the day sheet
- read the stock clipboard
- read the tickets on the rail
- write the chalkboard to `board.txt`

You cannot explain most of this yet. That is the point.
Run the next four cells in order, and keep a list of what surprises you.

**First, the café in three small files.** Run the next cell — it writes the fixtures to a temp folder; nothing prints.

In [2]:
# The whole game, part 1 of 4: the café's three files.
import tempfile
from shared.model_types import ModelResponse, ToolCall

wd = Path(tempfile.mkdtemp())
(wd / "day.txt").write_text(
    "06:30 Open + ovens on\n"
    "07:00 First regulars\n"
    "08:15 School-run rush — never let the queue pass 3\n"
    "09:00 Two bakes, one oven\n"
    "10:00 Mina's order (school group)\n"
    "12:00 Lunch rush\n"
    "15:00 Supplier delivery\n"
    "17:30 Close + count the till\n")
(wd / "stock.txt").write_text(
    "milk: low\n"
    "oat milk: 2 cartons\n"
    "coffee beans: 2 bags\n"
    "cocoa: plenty\n"
    "peanuts: sealed, store room only\n")
_ = (wd / "orders.txt").write_text(
    "#1 Sasha — oat-milk latte\n"
    "#2 Aldo — decaf + crossword\n"
    "#3 Mina + school group — cocoa (see red card)\n")



# The return values are file sizes — not output; the run below is the show.


**Tab's tools.** Run the next cell — it defines four small tools over those files; nothing prints.

In [3]:
# The whole game, part 2 of 4: Tab's four tools.
def write_file(path, content):
    (wd / path).write_text(content)
    return f"wrote {path}"


TOOLS = {
    "read_file": lambda path: (wd / path).read_text(),
    "read_stock": lambda: (wd / "stock.txt").read_text(),
    "first_tickets": lambda: (wd / "orders.txt").read_text(),
    "write_file": write_file,
}




**The chalkboard, and the write step.** Run the next cell — it defines how `board.txt` gets its line; nothing prints.

In [4]:
# The whole game, part 3 of 4: the chalkboard line, and the write step.
def board_line(day, stock, tickets):
    marks = [ln for ln in day.splitlines() if ln.strip()]
    rush = next(ln for ln in marks if "School-run rush" in ln)
    rush_time = rush.split(" ", 1)[0]
    low = [ln.split(":", 1)[0] for ln in stock.splitlines()
           if ln.rstrip().endswith(": low")]
    rail = [ln for ln in tickets.splitlines() if ln.strip()]
    return (f"open: ovens on · {len(marks)} day marks · first rush "
            f"{rush_time} · {len(rail)} tickets on the rail · "
            f"stock: {' + '.join(low)} low")


def write_step(messages, _tools):
    parts = [m["content"] for m in messages if m["role"] == "tool"]
    return ModelResponse(tool_calls=(ToolCall(
        id="call_write_file", name="write_file",
        arguments={"path": "board.txt",
                   "content": board_line(parts[0], parts[1],
                                         parts[2])}),))




**The run.** Run the next cell — you should see: a five-step trace ending `status completed · steps 5`, then the artifact line `open: ovens on · 8 day marks · first rush 08:15 · 3 tickets on the rail · stock: milk low`.

In [5]:
# The whole game, part 4 of 4: the run itself.
model = FakeModel([call("read_file", path="day.txt"),
                   call("read_stock"),
                   call("first_tickets"),
                   write_step, say("done")])
messages = [{"role": "user", "content": "Open the café."}]
for step in range(1, 9):  # a budget — Chapter 1 explains why
    response = model(messages)
    if response.is_final:
        print(f"step {step} · model → final {response.text!r}")
        print(f"status completed · steps {step}")
        break
    for tc in response.tool_calls:
        args = ", ".join(f"{k}={v!r}" for k, v in tc.arguments.items())
        print(f"step {step} · model → tool_call {tc.name}({args})")
        out = TOOLS[tc.name](**tc.arguments)
        print(f"step {step} · tool  → {out!r}")
        messages.append({"role": "tool", "tool_call_id": tc.id,
                         "name": tc.name, "content": out})
print("artifact board.txt:", (wd / "board.txt").read_text())


step 1 · model → tool_call read_file(path='day.txt')
step 1 · tool  → "06:30 Open + ovens on\n07:00 First regulars\n08:15 School-run rush — never let the queue pass 3\n09:00 Two bakes, one oven\n10:00 Mina's order (school group)\n12:00 Lunch rush\n15:00 Supplier delivery\n17:30 Close + count the till\n"
step 2 · model → tool_call read_stock()
step 2 · tool  → 'milk: low\noat milk: 2 cartons\ncoffee beans: 2 bags\ncocoa: plenty\npeanuts: sealed, store room only\n'
step 3 · model → tool_call first_tickets()
step 3 · tool  → '#1 Sasha — oat-milk latte\n#2 Aldo — decaf + crossword\n#3 Mina + school group — cocoa (see red card)\n'
step 4 · model → tool_call write_file(path='board.txt', content='open: ovens on · 8 day marks · first rush 08:15 · 3 tickets on the rail · stock: milk low')
step 4 · tool  → 'wrote board.txt'
step 5 · model → final 'done'
status completed · steps 5
artifact board.txt: open: ovens on · 8 day marks · first rush 08:15 · 3 tickets on the rail · stock: milk low


## What you can't explain yet

- Why the loop stopped at step 5 — and what stops a version that never answers → Chapter 1
- What a *schema* is, and how the model knew `read_file` takes a `path`, while `read_stock` and `first_tickets` take nothing at all → Chapter 2
- Where the chalkboard numbers came from — the 8 day marks, the first rush at 08:15, the 3 tickets: they arrived through three tool results → Chapter 2
- What happens when the conversation outgrows the model's window → Chapter 3
- What would stop this agent spending the café's money, or putting the wrong thing on the prep counter → Chapter 5

Keep your own list. By Chapter 11 you will explain every line of that run — and grade it.

Stage 1 of 4 — Watch it break, then predict · ~13 min

## The failure, shown

**Goal:** see the starter's two shape bugs with your own eyes — quiet bugs are why this chapter comes first.

The starter has two small bugs with large consequences:
- it sends the prompt as a bare string instead of the messages shape
- it hands the caller the whole response object instead of the answer

**Run the next cell** — you should see: what actually crosses the wire, and what never arrives.

In [6]:
model = FakeModel([say("4")])
answer = starter.ask(model, "What is 2+2?", system="Be terse.")
sent, _tools = model.calls[0]
print("messages sent :", sent, " <- a bare string, not a list of dicts")
print("system prompt : dropped — it never reached the model")
print("answer type   :", type(answer).__name__,
      " (the caller asked for the text '4')")

messages sent : What is 2+2?  <- a bare string, not a list of dicts
system prompt : dropped — it never reached the model
answer type   : ModelResponse  (the caller asked for the text '4')


Nothing crashed. The starter *looks* like it works — until a caller
tries to use the answer as text, or wonders why the system prompt
had no effect. Shape bugs are quiet like that, which is why this
chapter exists before any other.

## Predict

> **PREDICT**
> Before you build anything, call the shot.

The fixed version will run `ask(model, "hello", system="Be terse.")`.
Fill in the cell below: how many model calls happen, what role is
the first message in the list, and what Python type comes back?

In [7]:
# Your prediction — fill this in, then run the next cell.
prediction = {
    "model_calls": None,        # e.g. 1
    "first_message_role": "",   # e.g. "system"
    "answer_type": "",          # e.g. "str"
}

In [8]:
expected = {'model_calls': 1, 'first_message_role': 'system', 'answer_type': 'str'}
if any(v is None or v == "" for v in prediction.values()):
    print("No prediction yet — a guess in the cell above, "
          "then re-run me.")
else:
    right = [k for k in expected if prediction.get(k) == expected[k]]
    if len(right) == len(expected):
        print("Exactly right — hold onto that instinct; "
              "the build below is why it works.")
    else:
        missed = [k for k in expected if k not in right]
        print(f"Partway there. Still to rethink: {', '.join(missed)}.")
        print("Hint: one call only — and the system prompt is a message like any other, just first.")

No prediction yet — a guess in the cell above, then re-run me.


## Watch the board

This is the café board at Chapter 0's level: a bare counter and the chalkboard — no fixtures yet.
- Each chapter adds exactly one fixture: the layer you are building, made visible in the café.
- What Tab works with now: the chalkboard line, and the call itself.

**Run the next cell** — you should see: the board at its smallest.

In [9]:
# The café board at this chapter's level (shared/board.py).
from shared.board import render_board, render_board_html
try:
    from IPython.display import HTML, display
    display(HTML(render_board_html(0)))
except Exception:
    print(render_board(0))

**Stage 1 done — you now have:** the finished run in view, the starter's two bugs named, your prediction on record, and the board at Chapter 0's level.

Stage 2 of 4 — Build the layer · ~4 min

## The mechanism

**Goal:** build the two functions that fix both bugs — the shape the harness sends, and the text the caller gets back.

A request is a **list of messages**, each a dict with a `role` and a `content`. The system prompt, when there is one, goes first — it is the instruction every later message is read under.

And the call is *stateless*: the provider remembers nothing between calls. If a conversation is to continue, the harness keeps the list and resends it. That one fact forces almost every mechanism in the chapters ahead.

### Build — the messages shape, and an honest return

Two functions, no more. `build_messages` assembles the list; `ask` makes exactly one call and returns the response's *text*.

**Run the next cell** — it defines both functions; nothing prints.

In [10]:
def build_messages(prompt, system=None):
    """The messages shape: a list of {"role": ..., "content": ...}
    dicts. Order matters — it *is* the conversation."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    return messages


def ask(model, prompt, system=None):
    """Make exactly one model call and return its text."""
    response = model(build_messages(prompt, system))
    return response.text

**Run the next cell** — you should see: the sent list with the system dict first, the answer `'ok'`, and exactly 1 model call.

In [11]:
model = FakeModel([say("ok")])
answer = ask(model, "hello", system="Be terse.")
sent, _tools = model.calls[0]
print("messages sent:", sent)
print("answer       :", repr(answer), "· model calls:", model.call_count)
assert answer == "ok" and model.call_count == 1
assert sent[0] == {"role": "system", "content": "Be terse."}

messages sent: [{'role': 'system', 'content': 'Be terse.'}, {'role': 'user', 'content': 'hello'}]
answer       : 'ok' · model calls: 1


**Stage 2 done — you now have:** `build_messages` (the shape) and `ask` (one call, text back) — the whole layer in two functions.

Stage 3 of 4 — Sabotage and checkpoint · ~6 min

## Sabotage

> **SABOTAGE**
> Break it on purpose, so you recognise the breakage forever.

**Goal:** feel statelessness from the inside — a harness that forgets between calls fails a question any child could answer.

A forgetful harness sends only the newest message on every call — the cheapest possible "conversation".

**Run the next cell** — you should see: the forgetful harness answer `I don't know it.`, the history-keeping one answer `Your codeword is PLUM.`

In [12]:
def forgetful_ask(model, prompt):
    # Sends ONLY the newest message — no history, ever.
    return model([{"role": "user", "content": prompt}]).text


def step2(messages, _tools):
    seen = " ".join(m["content"] for m in messages)
    return say("Your codeword is PLUM.") if "PLUM" in seen \
        else say("I don't know it.")


m = FakeModel([say("Noted.")])
forgetful_ask(m, "My codeword is PLUM.")
m = FakeModel([step2])
print("forgetful harness       :", forgetful_ask(m, "What is my codeword?"))

history = [{"role": "user", "content": "My codeword is PLUM."},
           {"role": "assistant", "content": "Noted."},
           {"role": "user", "content": "What is my codeword?"}]
m = FakeModel([step2])
print("history-keeping harness :", m(history).text)

forgetful harness       : I don't know it.
history-keeping harness : Your codeword is PLUM.


## Checkpoint

> **CHECKPOINT**
> Same tests the repo chapter runs — no drift, no honour system.

This runs chapter 0's real checkpoint suite against the reference solution, in this notebook's own kernel. (Your notebook build is the same logic; the final cell re-checks its invariants directly.)

**Run the next cell** — you should see: five dots, `OK`, and `CHECKPOINT: PASS`.

In [13]:
# The chapter's real suite, run in this kernel (no subprocess).
import io, os, unittest, importlib.util

os.environ["HARNESS_VARIANT"] = "solution"
spec = importlib.util.spec_from_file_location(
    "nb_tests", ROOT / "chapters/00-raw-llm-call/tests/test_chapter00.py")
tmod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tmod)
suite = unittest.defaultTestLoader.loadTestsFromModule(tmod)
stream = io.StringIO()
result = unittest.TextTestRunner(stream=stream, verbosity=1).run(suite)
print(stream.getvalue().strip())
print("CHECKPOINT: PASS" if result.wasSuccessful() else "CHECKPOINT: NOT YET")
if not result.wasSuccessful():
    print("\nWhat this usually means:")
    for _tip in [
        "ask() handing back a ModelResponse instead of text: the caller wants response.text — unwrapping it is the harness's job.",
        "System prompt missing or not first: build_messages puts the system dict at index 0, before the user dict; anywhere else it is silently ignored.",
        "More than one model call, or tools offered: a raw call is exactly one model(messages) with tools=None. Looping is Chapter 1.",
    ]:
        print(" -", _tip)
assert result.wasSuccessful()

.....
----------------------------------------------------------------------
Ran 5 tests in 0.000s

OK
CHECKPOINT: PASS


**Stage 3 done — you now have:** proof, from the suite itself, that the call keeps its shape and its history.

Stage 4 of 4 — In the wild, and spot it · ~5 min

## In the wild

> **IN THE WILD**
> The raw call you just built is every product's atom.

**Goal:** place the call you built inside the real products — same shape, same statelessness, vendor-scale.

*Checked Oct 2026. **documented** = the vendor's docs say it; **inferred** = our reading; **not documented** = we looked.*

### The same call, in real products

- **Anthropic Messages API.** A request carries `model`,
  `max_tokens`, and a `messages` array of role/content objects;
  the system prompt is its own top-level field, not a message in
  the array. **documented** —
  https://docs.anthropic.com/en/docs/build-with-claude/tool-use
- **OpenAI APIs.** The same shape in mirror: an input list goes
  in, one response comes out, and multi-turn state is a list the
  *caller* maintains and resends — their function-calling guide
  builds that list by hand. **documented** —
  https://developers.openai.com/api/docs/guides/function-calling
- **Products are calls plus harness.** OpenAI's docs note that
  agent products like Deep Research can run for minutes from a
  Responses request started in the background — the call is the
  atom; the polling, tools, and synthesis around it are harness.
  **documented** —
  https://developers.openai.com/api/docs/guides/background
- **Statelessness is the default** in both: keep the conversation
  server-side only if you explicitly opt into a state feature.
  The harness owning the list is the normal case, not a quirk of
  this curriculum. **documented** — both links above.

## Spot it in the wild

> **SPOT IT**
> Two minutes, in the tools you already use.

In Claude Code, run `/clear` mid-session and ask about something
you discussed before it. The product simply starts over —
proof that the conversation lived in the harness, not the model.

Note in the cell below: what did it forget, and what (files,
memory) survived the clear?

In [14]:
# Your spot-it notes (plain comments are fine):
# what I observed:
# which mechanism from this chapter it was:
# one question it raised:

Stuck? Work the ladder — each step shows a little more.

<details>
<summary>Hint 1 — a nudge</summary>

Notice what the caller actually needs back: not the response object, but its text — and ask what has to be true of the list you send for a system prompt to exist in it at all.
</details>

<details>
<summary>Hint 2 — the shape of it</summary>

1. `build_messages(prompt, system)`: start with an empty list → append `{"role": "system", ...}` only if a system prompt was given → append `{"role": "user", ...}`.
2. `ask(model, prompt, system)`: build the list → call the model exactly once with it → return `response.text`, nothing else.
</details>

## Solution — open only after the checkpoint passes

<details>
<summary>Chapter reference code (from solution/harness.py)</summary>

The whole chapter — the two functions are the entire layer.

```python
def build_messages(prompt, system=None):
    """The messages shape: a list of {"role": ..., "content": ...} dicts.
    Order matters -- it *is* the conversation."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    return messages


def ask(model, prompt, system=None):
    """Make exactly one model call and return its text."""
    response = model(build_messages(prompt, system))
    return response.text
```
</details>

In [15]:
# Restart-and-run-all self-check.
checks = []
msgs = build_messages("hello", system="Be terse.")
checks.append(("messages are role/content dicts, system first",
               msgs == [{"role": "system", "content": "Be terse."},
                        {"role": "user", "content": "hello"}]))
m = FakeModel([say("4")])
checks.append(("ask returns the text, in exactly one call",
               ask(m, "2+2?") == "4" and m.call_count == 1))
m = FakeModel([say("ok")])
checks.append(("no tools are offered in a raw call",
               ask(m, "hi") == "ok" and not m.calls[0][1]))
for name, ok in checks:
    print(("  ok  " if ok else "FAIL  ") + name)
assert all(ok for _n, ok in checks)
print("Chapter 0 complete — one call, the right shape, the answer as text.")

  ok  messages are role/content dicts, system first
  ok  ask returns the text, in exactly one call
  ok  no tools are offered in a raw call
Chapter 0 complete — one call, the right shape, the answer as text.


**Stage 4 done — you now have:** the raw call, verified as the atom inside real products — and a self-check below that re-proves your build from scratch.

## Make it yours

Nothing to personalize yet — Tab has no memory until
chapter 4. But start the note that matters: name one regular
of your own — their usual, their seat, the thing Tab must
never get wrong about them — and one rule for your café that
must never be broken. You will pin the rule as a constraint
in chapter 3. Optional; no checkpoint reads this.

## Go deeper (optional)

- [LLM Powered Autonomous Agents](https://lilianweng.github.io/posts/2023-06-23-agent/) — the map of this whole territory, drawn before most of it had names.
- [Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents) — the case for simple, inspectable machinery like this course's over clever frameworks.


In [16]:
# Progress passport: stamp this chapter in my-progress.md,
# beside the notebooks. The file is yours — it is not part
# of the repo, and re-running never double-stamps.
from shared.passport import stamp

print(stamp(0, 'The model is stateless. The harness owns the conversation.'))


# My progress — Under the Harness

✓ Ch 0 — The model is stateless. The harness owns the conversation. — 2026-10-10
✓ Ch 1 — A loop is a budget, not a while-true. — 2026-10-10
✓ Ch 2 — Tools are contracts. The schema is the promise, and errors are data, not exceptions. — 2026-10-10
✓ Ch 3 — Compaction is lossy compression of your own instructions. Pin what must not be lost. — 2026-10-10
✓ Ch 4 — Memory is a filing decision. Store deliberately, load deliberately. — 2026-10-10
✓ Ch 5 — Permissions are a threat model, not a dialog box. — 2026-10-10
✓ Ch 6 — A subagent is a fresh context with a narrow job. Isolation is the feature. — 2026-10-10
✓ Ch 7 — If you can't replay the run, you can't debug the agent. — 2026-10-10
✓ Ch 8 — A protocol is a registry you don't own. Discover, don't hardcode. — 2026-10-10
✓ Ch 9 — A skill is context delivered just in time. Advertise cheap, load late. — 2026-10-10
✓ Ch 10 — One gate is not a defense. Contain in layers, and assume the model will be talked 